# 03 · Echte klimaatsensoren uitlezen via HTTP — en de THI berekenen

Tot nu toe werkten we met gesimuleerde sensoren. Nu lezen we **echte** draadloze temperatuur- en vochtsensoren uit, via het web. **Sensor.Community** (vroeger Luftdaten.info) is een burgerwetenschapsnetwerk met tienduizenden zelfgebouwde sensoren (fijnstof, temperatuur, RV) die hun metingen elke 2,5 minuten via **wifi en HTTP** naar een centrale server sturen. Ook rond Gent hangen er meer dan honderd. De server stelt alles open via een **HTTP-API** in JSON.

Dat is precies de keten uit de les: sensor → wifi → internet → server → *jij* via HTTP. En de sensoren zijn dezelfde types die in IoT-stalsensoren zitten: DHT22, BME280 en Sensirion SHT30/SHT31 (de SHT31 zit ook in de Dragino S31 uit notebook 01).

## 1. HTTP in drie regels

Een HTTP-*request* heeft een **methode** (`GET` = geef me, `POST` = hier is iets), een **URL** en eventueel **parameters** (achter het `?`). Het *antwoord* heeft een **statuscode** (200 OK, 404 niet gevonden, 429 te veel verzoeken, 500 serverfout) en een **body**, hier JSON.

Vergelijk met MQTT: bij HTTP vraag *jij* telkens, bij MQTT *duwt* de broker. HTTP past bij "geef me de laatste stand" en bij dashboards; MQTT past bij "verwittig me zodra er iets verandert".

In [ ]:
import requests, json, os, pathlib
import pandas as pd
import matplotlib.pyplot as plt

DATA = pathlib.Path("../data")          # offline reservekopieën (snapshot van 19 okt. 2026)

def haal_json(url, params=None, reserve=None):
    '''GET-verzoek; valt terug op een lokaal bestand als het netwerk niet lukt.'''
    try:
        r = requests.get(url, params=params, timeout=15, headers={"User-Agent": "ugent-i700301-practicum"})
        print("GET", r.url, "->", r.status_code)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        if reserve is None:
            raise
        print("Netwerk niet beschikbaar (", type(e).__name__, ") -> reservebestand", reserve.name)
        return json.load(open(reserve, encoding="utf-8"))

## 2. Alle temperatuur-/vochtsensoren in een straal van 30 km rond Gent

De API van Sensor.Community: `https://data.sensor.community/airrohr/v1/filter/<filters>` geeft de **laatste 5 minuten** aan metingen. Filters: `area=<lat>,<lon>,<km>`, `type=<sensortypes>`, `country=BE`, … Documentatie: <https://github.com/opendata-stuttgart/meta/wiki/APIs>.

In [ ]:
url = "https://data.sensor.community/airrohr/v1/filter/area=51.05,3.72,30&type=DHT22,BME280,SHT31,SHT30"
ruw = haal_json(url, reserve=DATA / "sensor_community_gent_30km_2026-10-19.json")
print(len(ruw), "metingen ontvangen")
print(json.dumps(ruw[0], indent=1)[:700])      # bekijk de structuur van één meting

Elke meting is een *genest* JSON-object: locatie, sensor(type) en een lijst `sensordatavalues` met één element per grootheid. Zo'n structuur zet je om naar een platte tabel (één rij per sensor, één kolom per grootheid) — dit "uitvlakken" is dagelijkse kost bij IoT-data.

In [ ]:
rijen = []
for m in ruw:
    rij = {
        "sensor_id":  m["sensor"]["id"],
        "type":       m["sensor"]["sensor_type"]["name"],
        "lat":        float(m["location"]["latitude"]),
        "lon":        float(m["location"]["longitude"]),
        "binnen":     bool(m["location"]["indoor"]),
        "tijd":       pd.to_datetime(m["timestamp"], utc=True),
    }
    for v in m["sensordatavalues"]:
        rij[v["value_type"]] = float(v["value"])     # temperature, humidity, (pressure)
    rijen.append(rij)

df = pd.DataFrame(rijen).drop_duplicates("sensor_id", keep="last")
df = df[df["temperature"].between(-30, 60) & df["humidity"].between(0, 100)]   # grove fouten eruit
print(df["type"].value_counts().to_string())
df.head()

In [ ]:
print(df[["temperature", "humidity"]].describe().round(1))
fig, ax = plt.subplots(figsize=(5, 4))
for t, g in df.groupby("type"):
    ax.scatter(g["temperature"], g["humidity"], label=f"{t} (n={len(g)})", s=18, alpha=0.7)
ax.set_xlabel("temperatuur (°C)"); ax.set_ylabel("RV (%)"); ax.legend()
ax.set_title("Alle sensoren rond Gent, laatste 5 minuten"); plt.tight_layout(); plt.show()

### Opdracht 2.1
Waarom staan er punten bij 99,9 % RV, en welk sensortype is dat vooral? (Tip: zoek de specificaties van de DHT22 en vergelijk met de SHT31 uit de Dragino-handleiding, §1.3.) Wat betekent dat voor een sensor in een stal met hoge luchtvochtigheid?

In [ ]:
# Opdracht 2.1 — controle: welk type zit op 99,9 %?
print(df[df["humidity"] >= 99.9]["type"].value_counts())
print(df.groupby("type")["humidity"].describe().round(1))

<details>
<summary><b>Oplossing tonen</b></summary>

Bijna alle punten op 99,9 % zijn **DHT22**-sensoren. De DHT22 is een goedkope (± € 5) sensor met ±2–5 % RV nauwkeurigheid die bij hoge vochtigheid verzadigt en na condensatie lang op 99,9 % blijft "hangen". De **SHT31** (in de Dragino S31, handleiding §1.3) geeft ±2 % RV over 0–100 %, is gekalibreerd en zit in een anticondensbehuizing. In een stal met vaak > 80 % RV is een DHT22 dus onbetrouwbaar voor RV, en dus ook voor de THI.

</details>

## 3. De sensor het dichtst bij de campus

We berekenen de afstand van elke sensor tot een punt (standaard: campus Coupure, Gent; verander gerust naar je eigen gemeente) en nemen de dichtstbijzijnde.

In [ ]:
import numpy as np

MIJN_LAT, MIJN_LON = 51.053, 3.706        # Campus Coupure, Gent  (ILVO Melle: 50.990, 3.785)

def afstand_km(lat1, lon1, lat2, lon2):
    '''Haversine-afstand in km.'''
    p = np.pi / 180
    a = (np.sin((lat2-lat1)*p/2)**2
         + np.cos(lat1*p) * np.cos(lat2*p) * np.sin((lon2-lon1)*p/2)**2)
    return 2 * 6371 * np.arcsin(np.sqrt(a))

df["afstand_km"] = afstand_km(MIJN_LAT, MIJN_LON, df["lat"], df["lon"]).round(1)
dichtstbij = df.sort_values("afstand_km").head(5)
dichtstbij[["sensor_id", "type", "afstand_km", "temperature", "humidity", "tijd"]]

## 4. Vergelijken met een weermodel (Open-Meteo)

Open-Meteo geeft gratis, zonder sleutel, de actuele waarden van een weermodel voor elk punt. Een lokale sensor meet *op die plek* (zon, muur, stad); het model geeft een gemiddelde over een rooster van enkele km. Het verschil leert je iets over de waarde van een *eigen* sensor.

In [ ]:
s = dichtstbij.iloc[0]
try:
    meteo = haal_json("https://api.open-meteo.com/v1/forecast",
                      params={"latitude": s["lat"], "longitude": s["lon"],
                              "current": "temperature_2m,relative_humidity_2m"})
    h = meteo["current"]
    print(f"Sensor {s['sensor_id']} ({s['type']}, {s['afstand_km']} km): {s['temperature']} °C / {s['humidity']} % om {s['tijd']:%H:%M} UTC")
    print(f"Open-Meteo op dezelfde plek:            {h['temperature_2m']} °C / {h['relative_humidity_2m']} % om {h['time']} UTC")
except Exception as e:
    print("Open-Meteo niet bereikbaar:", e)

## 5. Van meting naar kengetal: de THI

In les 2 zag je de **Temperature-Humidity Index**. Een veelgebruikte vorm (NRC, 1971) is

$$\mathrm{THI} = 0{,}8\,T + \frac{RV}{100}\,(T - 14{,}4) + 46{,}4$$

met $T$ in °C en $RV$ in %. Voor melkvee gelden vaak deze klassen: < 68 geen stress · 68–71 mild · 72–79 matig · 80–89 ernstig · ≥ 90 noodsituatie. (Andere diersoorten en bronnen gebruiken andere grenzen; controleer de formule en de klassen met je cursus.)

Dit is wat een IoT-platform of klimaatcomputer doet: uit twee ruwe metingen een kengetal maken en daar drempels op zetten.

In [ ]:
def thi(T, RV):
    return 0.8 * T + (RV / 100) * (T - 14.4) + 46.4

def thi_klasse(x):
    if x < 68:  return "geen stress"
    if x < 72:  return "mild"
    if x < 80:  return "matig"
    if x < 90:  return "ernstig"
    return "noodsituatie"

df["THI"] = thi(df["temperature"], df["humidity"]).round(1)
df["THI_klasse"] = df["THI"].apply(thi_klasse)
print(df["THI_klasse"].value_counts().to_string())
df.sort_values("THI", ascending=False)[["sensor_id", "type", "temperature", "humidity", "THI", "THI_klasse"]].head()

## 6. Een dag uit het leven van één sensor

De live-API geeft maar 5 minuten. Het **archief** (`https://archive.sensor.community/<datum>/<datum>_<type>_sensor_<id>.csv`) geeft per sensor en per dag een CSV met een meting om de ~2,5 minuten. We nemen sensor **82001**, een Sensirion SHT30 ten zuidwesten van Gent — hetzelfde sensorfamilielid als in de Dragino S31.

In [ ]:
import io, datetime as dt

SENSOR_ID, SENSOR_TYPE = 82001, "sht30"
datum = (dt.date.today() - dt.timedelta(days=1)).isoformat()      # gisteren (vandaag is nog niet afgesloten)
url_csv = f"https://archive.sensor.community/{datum}/{datum}_{SENSOR_TYPE}_sensor_{SENSOR_ID}.csv"
try:
    r = requests.get(url_csv, timeout=20); r.raise_for_status()
    dag = pd.read_csv(io.StringIO(r.text), sep=";")
    print("GET", url_csv, "->", r.status_code)
except Exception as e:
    print("Archief niet bereikbaar (", type(e).__name__, ") -> reservebestand van 18 okt. 2026")
    dag = pd.read_csv(DATA / "2026-10-18_sht30_sensor_82001.csv", sep=";")

dag["timestamp"] = pd.to_datetime(dag["timestamp"], utc=True)
dag["THI"] = thi(dag["temperature"], dag["humidity"])
print(len(dag), "metingen; interval ≈", dag["timestamp"].diff().median())
dag.head(3)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 7), sharex=True)
axes[0].plot(dag["timestamp"], dag["temperature"]); axes[0].set_ylabel("°C")
axes[1].plot(dag["timestamp"], dag["humidity"], color="tab:gray"); axes[1].set_ylabel("% RV")
axes[2].plot(dag["timestamp"], dag["THI"], color="tab:red"); axes[2].set_ylabel("THI")
for y, lbl in [(68, "mild"), (72, "matig")]:
    axes[2].axhline(y, linestyle="--", linewidth=1, color="k"); axes[2].text(dag["timestamp"].iloc[0], y + 0.3, lbl, fontsize=8)
axes[0].set_title(f"Sensor {SENSOR_ID} (SHT30), {dag['timestamp'].dt.date.iloc[0]}")
fig.autofmt_xdate(); plt.tight_layout(); plt.show()

## 7. Opdrachten

1. **Jouw gemeente.** Zet `MIJN_LAT, MIJN_LON` op je eigen woonplaats en zoek de dichtstbijzijnde sensor. Hoe ver is die? Zou je op basis van die sensor een stalklimaat mogen beoordelen? Waarom (niet)?
2. **Binnen of buiten?** Filter `df` op `binnen == True`. Hoeveel binnensensoren zijn er, en verschilt hun THI-verdeling van de buitensensoren?
3. **Bemonsteringsinterval.** De sensor meet om de 2,5 min; een Dragino op batterij meestal om de 10–20 min. Herbemonster `dag` naar 20 minuten (`dag.resample("20min", on="timestamp").mean()`) en vergelijk de maximale THI met die van de volledige reeks. Wat verlies je?
4. **HTTP vs MQTT.** Sensor.Community-sensoren sturen elke 2,5 min een HTTP POST met JSON. Beredeneer wat er zou veranderen als ze MQTT zouden gebruiken: voor de sensor, voor de server, en voor jou als gebruiker van de data.
5. **Foutieve waarden.** We gooiden metingen buiten −30…60 °C en 0…100 % weg. Welke *andere* controles zou een klimaatcomputer moeten doen voor ze een meting gebruikt om de ventilatie te sturen? (Denk aan: tijdstip, plotse sprongen, bevroren waarden, batterijspanning.)

In [ ]:
# Opdracht 2 en 3 — controle
print("binnen:", (df["binnen"]).sum(), "buiten:", (~df["binnen"]).sum())
print(df.groupby("binnen")["THI"].describe().round(1))

dag20 = dag.resample("20min", on="timestamp").mean(numeric_only=True)
print(f"max THI volledige reeks: {dag['THI'].max():.1f}   max THI na 20-min-gemiddelde: {dag20['THI'].max():.1f}")

<details>
<summary><b>Oplossing tonen</b></summary>

1. Hangt van je gemeente af. Een sensor op enkele km zegt iets over het *buiten*klimaat, niets over het stalklimaat: de stal heeft eigen warmteproductie (dieren), ventilatie en vochtbronnen. Voor THI-sturing heb je sensoren *in* de afdeling nodig; buitensensoren helpen wel om de ventilatie voorspellend bij te sturen.
2. Meestal een handvol binnensensoren; hun RV ligt lager en hun THI hoger en stabieler dan buiten.
3. Herbemonsteren naar 20 min vlakt de pieken af: de maximale THI daalt (enkele tienden tot ruim een punt). Voor een alarm op een drempel telt de piek, dus een kort interval; voor dagrapportage volstaat 20 min.
4. Sensor: één permanente verbinding i.p.v. elke 2,5 min een nieuwe TCP/TLS-handshake (minder energie en latentie). Server: een broker i.p.v. webserver + database-API, schaalbaar naar veel abonnees. Gebruiker: live meekijken i.p.v. pollen; maar "geef me de laatste 24 u" vereist een extra opslaglaag, daar blijft HTTP handig.
5. Tijdstempel ouder dan x minuten (verbinding weg → niet sturen op oude data); sprong > x °C per minuut (storing); identieke waarde gedurende uren (bevroren sensor); batterij onder drempel; waarden buiten het meetbereik van het sensortype; vergelijking met een tweede sensor in dezelfde afdeling (redundantie).

</details>